In [2]:
from sklearn.datasets import fetch_lfw_people

lfw = fetch_lfw_people(min_faces_per_person=2, resize=0.5)
print("Images shape:", lfw.images.shape)
print("Number of identities:", len(lfw.target_names))
print("First 10 names:", lfw.target_names[:10])

Images shape: (9164, 62, 47)
Number of identities: 1680
First 10 names: ['Aaron Peirsol' 'Aaron Sorkin' 'Abdel Nasser Assidi' 'Abdoulaye Wade'
 'Abdullah' 'Abdullah Gul' 'Abdullah al-Attiyah' 'Abdullatif Sener'
 'Abel Pacheco' 'Abid Hamid Mahmud Al-Tikriti']


In [6]:
import os
import random
from PIL import Image
import numpy as np

output_dir = "../data/public/lfw_subset"
os.makedirs(output_dir, exist_ok=True)

num_identities = 25

# get all available identity indices
all_indices = list(range(len(lfw.target_names)))

# randomly pick identities
selected_indices = random.sample(all_indices, num_identities)

saved_count = 0

for person_index in selected_indices:
    person_name = lfw.target_names[person_index]
    person_folder = os.path.join(output_dir, person_name.replace(" ", "_"))
    os.makedirs(person_folder, exist_ok=True)

    # find all images for this person
    indices = np.where(lfw.target == person_index)[0]

    for img_num, idx in enumerate(indices):
        img = lfw.images[idx]

        img_uint8 = (img * 255).astype(np.uint8) if img.max() <= 1 else img.astype(np.uint8)
        img_pil = Image.fromarray(img_uint8)

        save_path = os.path.join(
            person_folder,
            f"{person_name.replace(' ', '_')}_{img_num+1:02d}.jpg"
        )
        img_pil.save(save_path)

        saved_count += 1

print("Saved images:", saved_count)
print("Selected identities:", [lfw.target_names[i] for i in selected_indices])

Saved images: 117
Selected identities: [np.str_('Tom Daschle'), np.str_('Tony Curtis'), np.str_('Arnaud Clement'), np.str_('Felipe Perez Roque'), np.str_('Amanda Beard'), np.str_('William Hochul'), np.str_('Ishaq Shahryar'), np.str_('Jorge Batlle'), np.str_('Gloria Allred'), np.str_('Barbra Streisand'), np.str_('Ellen DeGeneres'), np.str_('Muhammad Ali'), np.str_('Sergey Lavrov'), np.str_('Isabelle Huppert'), np.str_('Elton John'), np.str_('Richard Crenna'), np.str_('Maria Soledad Alvear Valenzuela'), np.str_('Johnny Unitas'), np.str_('Ralf Schumacher'), np.str_('Lawrence MacAulay'), np.str_('Chung Mong-hun'), np.str_('Gil de Ferran'), np.str_('Christine Gregoire'), np.str_('Garry Kasparov'), np.str_('Susan Sarandon')]


In [7]:
import shutil

random.seed(67)

lfw_dir = "../data/public/lfw_subset"
unauth_dir = "../data/test/unauthorized"

os.makedirs(unauth_dir, exist_ok=True)

person_folders = [
    folder for folder in os.listdir(lfw_dir)
    if os.path.isdir(os.path.join(lfw_dir, folder))
]

person_folders.sort()

copied = 0

for i, person in enumerate(person_folders, start=1):
    person_path = os.path.join(lfw_dir, person)

    image_files = [
        f for f in os.listdir(person_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if not image_files:
        continue

    chosen_file = random.choice(image_files)

    src = os.path.join(person_path, chosen_file)
    dst = os.path.join(unauth_dir, f"lfw_unknown_{i:02d}.jpg")

    shutil.copy(src, dst)
    copied += 1

print(f"Copied {copied} LFW images into unauthorized/")

Copied 25 LFW images into unauthorized/


In [ ]:
import kagglehub

download_path = kagglehub.dataset_download("vasukipatel/face-recognition-dataset")

print("Downloaded to:", download_path)

target_dir = "../data/public/kaggle_face_dataset_raw"

for item in os.listdir(download_path):
    src_path = os.path.join(download_path, item)
    dst_path = os.path.join(target_dir, item)
    
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
    else:
        shutil.copy2(src_path, dst_path)

print("Dataset copied to:", target_dir)

/home/taming3joy/Documents/Coding/CMKL/Year 1/Semester 2/SEC-301/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 726M/726M [00:37<00:00, 20.1MB/s] 

Extracting files...


Downloaded to: /home/taming3joy/.cache/kagglehub/datasets/vasukipatel/face-recognition-dataset/versions/1
Dataset copied to: ../data/public/kaggle_face_dataset_raw


In [10]:
from collections import defaultdict

source_dir = "../data/public/kaggle_face_dataset_raw/Faces/Faces"
subset_dir = "../data/public/kaggle_subset"

os.makedirs(subset_dir, exist_ok=True)

# Step 1: group images by person name
person_to_files = defaultdict(list)

for file_name in os.listdir(source_dir):
    if not file_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    # Split at the last underscore only
    if "_" not in file_name:
        continue

    person_name = file_name.rsplit("_", 1)[0].strip()
    person_to_files[person_name].append(file_name)

# Step 2: keep only people with at least 1 image
all_people = list(person_to_files.keys())

# Step 3: choose 25 random people
selected_people = random.sample(all_people, 25)

# Step 4: create folders and copy images
total_copied = 0

for person in selected_people:
    safe_folder_name = person.replace(" ", "_")
    person_folder = os.path.join(subset_dir, safe_folder_name)
    os.makedirs(person_folder, exist_ok=True)

    for i, file_name in enumerate(sorted(person_to_files[person]), start=1):
        src = os.path.join(source_dir, file_name)
        ext = os.path.splitext(file_name)[1].lower()
        dst = os.path.join(person_folder, f"{safe_folder_name}_{i:02d}{ext}")
        shutil.copy2(src, dst)
        total_copied += 1

print("Selected people:")
for person in selected_people:
    print("-", person)

print(f"\nCreated subset at: {subset_dir}")
print(f"Total copied images: {total_copied}")

Selected people:
- Priyanka Chopra
- Brad Pitt
- Virat Kohli
- Vijay Deverakonda
- Dwayne Johnson
- Camila Cabello
- Claire Holt
- Natalie Portman
- Henry Cavill
- Kashyap
- Anushka Sharma
- Margot Robbie
- Andy Samberg
- Charlize Theron
- Zac Efron
- Tom Cruise
- Courtney Cox
- Roger Federer
- Akshay Kumar
- Amitabh Bachchan
- Ellen Degeneres
- Elizabeth Olsen
- Alia Bhatt
- Jessica Alba
- Hrithik Roshan

Created subset at: ../data/public/kaggle_subset
Total copied images: 2045


In [11]:
subset_dir = "../data/public/kaggle_subset"
unauth_dir = "../data/test/unauthorized"

os.makedirs(unauth_dir, exist_ok=True)

person_folders = [
    folder for folder in os.listdir(subset_dir)
    if os.path.isdir(os.path.join(subset_dir, folder))
]

person_folders.sort()

copied = 0

for i, person in enumerate(person_folders, start=1):
    person_path = os.path.join(subset_dir, person)

    image_files = [
        f for f in os.listdir(person_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if not image_files:
        continue

    chosen_file = random.choice(image_files)

    src = os.path.join(person_path, chosen_file)
    dst = os.path.join(unauth_dir, f"kaggle_unknown_{i:02d}.jpg")

    shutil.copy2(src, dst)
    copied += 1

print(f"Copied {copied} Kaggle images into unauthorized/")

Copied 25 Kaggle images into unauthorized/
